# LoveDA backbone-pretrained baselines — part 2

Backbone-pretrained mode. Each model uses the ImageNet backbone initialization declared by its pinned public config; segmentation heads are random and no trained segmentation checkpoint is loaded.

In [ ]:
from pathlib import Path

TRAIN_ROOT = Path('/kaggle/input/datasets/alienxc137/loveda-satellite-images-semantic-segmentation/Train')
VAL_ROOT = Path('/kaggle/input/datasets/alienxc137/loveda-satellite-images-semantic-segmentation/Val')
LANDCOVER = Path('/kaggle/working/LandCover')
MMSEG = Path('/kaggle/working/mmsegmentation')
DATA_ROOT = Path('/kaggle/working/loveda_mmseg')
OUTPUT = Path('/kaggle/working/loveda_backbone_pretrained_part2')
MODELS = ['segmenter', 'swin_upernet', 'convnext', 'poolformer']
SEED = 2025
print('Models:', MODELS)

In [ ]:
import subprocess, sys

def run(command):
    print('>', ' '.join(map(str, command)))
    subprocess.check_call([str(x) for x in command])

if not (LANDCOVER / '.git').exists():
    run(['git', 'clone', 'https://github.com/TuanHungGiang/LandCover.git', LANDCOVER])
else:
    run(['git', '-C', LANDCOVER, 'pull', '--ff-only'])
if not (MMSEG / '.git').exists():
    run(['git', 'clone', 'https://github.com/open-mmlab/mmsegmentation.git', MMSEG])
run(['git', '-C', MMSEG, 'checkout', '--detach', 'b040e147adfa027bbc071b624bedf0ae84dfc922'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'mmengine>=0.10,<1.0', 'mmcv-lite>=2.0.0rc4,<2.2.0', 'mmpretrain>=1.0,<2.0'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', MMSEG])


In [ ]:
PIPELINE = LANDCOVER / 'baseline_model' / 'paper_faithful'
run([sys.executable, PIPELINE / 'prepare_loveda.py', '--train', TRAIN_ROOT, '--val', VAL_ROOT, '--output', DATA_ROOT])

In [ ]:
import torch
command = [sys.executable, PIPELINE / 'run_queue.py', '--models', *MODELS, '--mmseg-root', MMSEG, '--data-root', DATA_ROOT, '--output', OUTPUT, '--seed', str(SEED), '--gpus', str(max(1, torch.cuda.device_count()))]
run(command)

In [ ]:
import json, shutil
from IPython.display import display
import pandas as pd

summary_path = OUTPUT / 'summary.json'
if summary_path.exists():
    rows = json.loads(summary_path.read_text())
    display(pd.json_normalize(rows))
    archive = shutil.make_archive(str(OUTPUT), 'zip', root_dir=OUTPUT)
    print('Archive:', archive)
else:
    print('No summary yet. Inspect per-model status.json and logs in', OUTPUT)